# 26 · GSE135779 · scRNA_seq · are the GSE65391 modules present here?

Reads this study's `expression.rds` (the 33 children with SLE, because the GSE65391 modules come from SLE patients) and the GSE65391 end product `module_genes.csv` (gene names and
module labels only; no GSE65391 expression values). Writes `data/run_artifacts/GSE135779/array_module_preservation.csv`.

Module preservation (Langfelder P et al. *PLoS Comput Biol* 2011;7:e1001057) asks whether genes that
form a module in one study also hang together in another. The full method also needs the reference
study's expression data. Here only the gene lists travel, so we use the two **density** statistics,
which need only this study's data:

| statistic | meaning |
|---|---|
| mean_cor | mean Pearson correlation over all pairs of the module's genes, in this study |
| prop_var_PC1 | share of the module genes' variance explained by their first principal component |

**Test.** For each array module, the same statistics for 1,000 random gene sets of the same size drawn
from this study's genes. Z = (observed − mean of random) / SD of random. **Reading**, borrowed from
Langfelder 2011 for Zsummary: Z > 10 strong evidence, 2–10 weak to moderate, < 2 none.

In [1]:
source("../src/paths.R")
x  <- readRDS(art("GSE135779", "expression.rds"))
mg <- read.csv(art("GSE65391", "module_genes.csv"))
Zs <- scale(t(x$E[, x$meta$sle == 1]))       # the 33 children with SLE; each gene standardised in this study
Zs <- Zs[, colSums(is.na(Zs)) == 0]
c(samples = nrow(Zs), genes = ncol(Zs))

samples   genes 
     33   15353

In [2]:
stat <- function(Z) {
  p <- ncol(Z)
  c(mean_cor = (var(rowSums(Z)) - p) / (p * (p - 1)),        # sum of all pairwise correlations of standardised genes
    prop_var_PC1 = svd(Z, nu = 0, nv = 0)$d[1]^2 / sum(svd(Z, nu = 0, nv = 0)$d^2))
}
set.seed(SEED)
pres <- do.call(rbind, lapply(split(mg$gene, mg$module), function(g) {
  g_here <- intersect(g, colnames(Zs))
  obs  <- stat(Zs[, g_here, drop = FALSE])
  null <- replicate(1000, stat(Zs[, sample(colnames(Zs), length(g_here)), drop = FALSE]))
  z    <- (obs - rowMeans(null)) / apply(null, 1, sd)
  data.frame(genes_in_array_module = length(g), genes_measured_here = length(g_here),
             mean_cor = obs[1], Z_mean_cor = z[1], prop_var_PC1 = obs[2], Z_prop_var_PC1 = z[2])
}))
pres$module <- rownames(pres)
pres$reading <- cut(pmin(pres$Z_mean_cor, pres$Z_prop_var_PC1), c(-Inf, 2, 10, Inf),
                    labels = c("none", "weak to moderate", "strong"))
pres <- pres[order(-pres$Z_mean_cor), c("module", "genes_in_array_module", "genes_measured_here", "mean_cor",
                                          "Z_mean_cor", "prop_var_PC1", "Z_prop_var_PC1", "reading")]
format(pres, digits = 3)

,module,genes_in_array_module,genes_measured_here,mean_cor,Z_mean_cor,prop_var_PC1,Z_prop_var_PC1,reading
,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>,<I<chr>>
black,black,250,90,0.7534,95.97,0.811,35.859,strong
turquoise,turquoise,3013,2809,0.0978,87.89,0.220,12.259,strong
blue,blue,942,848,0.1902,83.15,0.339,24.918,strong
purple,purple,94,93,0.5416,72.88,0.607,23.762,strong
pink,pink,247,233,0.2986,68.98,0.426,20.903,strong
greenyellow,greenyellow,84,63,0.5067,48.82,0.559,17.464,strong
cyan,cyan,47,44,0.6010,47.89,0.639,18.371,strong
red,red,268,217,0.2041,44.47,0.284,8.214,weak to moderate
tan,tan,55,51,0.4229,35.92,0.503,13.230,strong


**Result.** By both statistics (the reading uses the smaller Z):
- **strong:** black, turquoise, blue, purple, pink, greenyellow, cyan, tan, midnightblue;
- **weak to moderate:** red, brown, yellow, magenta, salmon;
- **none:** lightcyan and green.

lightcyan is the neutrophil-granule module, and brown the red-cell module. Density-gradient PBMC
preparation removes most neutrophils and red cells, so these two are expected to be weak here.

In [3]:
write.csv(pres, art("GSE135779", "array_module_preservation.csv"), row.names = FALSE)